# Stage-2 scenario C — post-return quick look (prompt 27 T4)

First look at `stage2_C_n0` (16 snapped-Sobol rows, d = 6 lattice) and
`stage2_backfill_C` (nuclear full-lattice + pv top-up), collected by the chained
`collect_stage2.sh` automation (commit `fc98902`, stage2-bot). Includes the
pre-registered GP LOO diagnostics for the 16→32 top-up decision.

**Caveat carried on every nuclear number:** g1 fuel-deletion APPROX (patch pending
PI review — not implemented).

In [1]:
import json, os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"axes.labelweight": "bold", "axes.titleweight": "bold",
                     "font.weight": "bold"})   # house style: bold axes

CAMPAIGN = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, CAMPAIGN)
from tiers import TIERS, STAGE2_LATTICE, load_gen_pmax

OBJ_COLS = ["load_shed_mwh", "true_curtailment_mwh", "total_cost_raw_usd",
            "total_cost_less_synthetic_usd", "reserve_shortfall_mwh", "thermal_starts"]
FLOORS = {"load_shed_mwh": 3000.0, "true_curtailment_mwh": 5000.0,
          "total_cost_raw_usd": 0.5e6, "total_cost_less_synthetic_usd": 0.5e6,
          "reserve_shortfall_mwh": None, "thermal_starts": None}  # pi_0911 §3.5
_PMAX = load_gen_pmax()
NAMEPLATE = {t: sum(_PMAX[m] for m in TIERS[t]["members"]) for t in TIERS}
SORTED_TIERS = sorted(TIERS)

def load_wave(name):
    wdir = os.path.join(CAMPAIGN, "waves", name)
    dm = pd.read_csv(os.path.join(wdir, "design_matrix.csv"))
    ob = pd.read_csv(os.path.join(wdir, "objectives.csv"))
    return dm.merge(ob[["index"] + OBJ_COLS], on="index", validate="1:1")

n0 = load_wave("stage2_C_n0")
bf = load_wave("stage2_backfill_C")
hull = load_wave("contour_303x317_C")
swC = load_wave("sweep_C")
assert len(n0) == 16 and len(bf) == 10 and len(hull) == 81
manifest = json.load(open(os.path.join(CAMPAIGN, "waves", "stage2_C_n0", "manifest.json")))
print("n0 sobol record:", manifest["sobol"])
os.makedirs("figs", exist_ok=True)

n0 sobol record: {'seed': 20260821, 'skip': 0, 'n': 16, 'n_drawn_total': 16, 'lattice': [0.05, 0.16875, 0.2875, 0.40625, 0.525, 0.64375, 0.7625, 0.88125, 1.0], 'scipy_version': '1.16.3'}


## 1. n₀ objective ranges vs the wind-pair grid hull

First glimpse of what the 4 extra tiers add beyond the (wind_303, wind_317) plane.
The x-axis is total PEM MW (pinned conversion, Σ ω·nameplate).

In [2]:
def total_mw(df):
    out = np.zeros(len(df))
    for t in SORTED_TIERS:
        out += df[f"{t}_omega"].fillna(0.0).to_numpy() * NAMEPLATE[t]
    return out

n0["total_mw"], hull["total_mw"] = total_mw(n0), total_mw(hull)
print(f"total PEM MW: n0 [{n0.total_mw.min():.0f}, {n0.total_mw.max():.0f}], "
      f"wind-pair grid [{hull.total_mw.min():.0f}, {hull.total_mw.max():.0f}]")

rows = []
for obj in OBJ_COLS:
    rows.append({"objective": obj,
                 "hull_min": hull[obj].min(), "hull_max": hull[obj].max(),
                 "n0_min": n0[obj].min(), "n0_max": n0[obj].max(),
                 "n0_below_hull_min": bool(n0[obj].min() < hull[obj].min()),
                 "n0_above_hull_max": bool(n0[obj].max() > hull[obj].max())})
ranges = pd.DataFrame(rows)
ranges.to_csv("n0_vs_hull_ranges.csv", index=False)
with pd.option_context("display.width", 200):
    print(ranges.round(1).to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, obj in zip(axes.flat, OBJ_COLS):
    ax.scatter(hull.total_mw, hull[obj], s=12, c="0.7", label="wind-pair grid (81)")
    ax.scatter(n0.total_mw, n0[obj], s=40, c="tab:red", marker="D", label="stage2 n0 (16)")
    ax.set_xlabel("total PEM MW"); ax.set_ylabel(obj, fontsize=8)
    ax.set_title(obj, fontsize=10)
axes.flat[0].legend(fontsize=8)
fig.suptitle("n0 (6-tier lattice) vs the wind-pair 9x9 grid", fontweight="bold")
fig.tight_layout(); fig.savefig("figs/n0_vs_hull.png", dpi=150); plt.close(fig)

total PEM MW: n0 [853, 2393], wind-pair grid [82, 1646]
                    objective    hull_min    hull_max      n0_min      n0_max  n0_below_hull_min  n0_above_hull_max
                load_shed_mwh      2920.2     40102.5        22.4     13729.3               True              False
         true_curtailment_mwh    470809.1   1494291.7     76268.9    616342.4               True              False
           total_cost_raw_usd 528327810.7 578550575.0 546956562.0 647870883.6              False               True
total_cost_less_synthetic_usd 526484930.8 572048842.7 539251294.1 641212849.2              False               True
        reserve_shortfall_mwh     43220.9    190437.4      7721.6     59038.1               True              False
               thermal_starts      2938.0      5000.0      2024.0      3135.0               True              False


## 2. Nuclear back-fill: does the trend break past ω = 0.5?

`sweep_C` nuclear OAT covers ω ∈ [0.05, 0.5] on the old 9-level grid; the back-fill
adds the 8 new lattice levels up to 1.0 (first-ever thermal ω > 0.5 production data —
09-19 smoke JID 1458448 validated the regime). Break test: OLS line fit on the ω ≤ 0.5
sweep points, residuals of the back-fill points from its extrapolation, in floor units
where a floor exists.

In [3]:
def oat_rows(df, tier):
    m = df[f"{tier}_omega"].notna()
    for t in SORTED_TIERS:
        if t != tier:
            m &= df[f"{t}_omega"].isna()
    return df[m].sort_values(f"{tier}_omega")

nuc_old = oat_rows(swC, "nuclear")          # 9 pts, omega 0.05..0.5
nuc_new = oat_rows(bf, "nuclear")           # 8 pts, lattice[1:]
pv_old = oat_rows(swC, "pv")                # 9 pts, omega 0.02..0.8
pv_new = oat_rows(bf, "pv")                 # 2 pts, 0.88125 / 1.0
assert len(nuc_old) == 9 and len(nuc_new) == 8 and len(pv_new) == 2

break_rows = []
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, obj in zip(axes.flat, OBJ_COLS):
    w_o, y_o = nuc_old["nuclear_omega"].to_numpy(), nuc_old[obj].to_numpy()
    w_n, y_n = nuc_new["nuclear_omega"].to_numpy(), nuc_new[obj].to_numpy()
    b, a = np.polyfit(w_o, y_o, 1)          # line on the old (≤0.5) regime
    grid = np.linspace(0.05, 1.0, 100)
    ax.plot(w_o, y_o, "o-", color="tab:blue", label="sweep_C (ω ≤ 0.5)")
    ax.plot(w_n, y_n, "D", color="tab:red", label="backfill (new lattice)")
    ax.plot(grid, b * grid + a, ":", color="0.4", label="linear fit ≤ 0.5, extrapolated")
    ax.set_xlabel("nuclear ω"); ax.set_ylabel(obj, fontsize=8); ax.set_title(obj, fontsize=10)
    resid = y_n - (b * w_n + a)
    above = resid[w_n > 0.5]
    row = {"objective": obj, "max_abs_resid_above_0p5": float(np.max(np.abs(above))),
           "signed_resid_at_1p0": float(resid[np.argmax(w_n)])}
    if FLOORS[obj]:
        row["max_resid_over_floor"] = float(np.max(np.abs(above)) / FLOORS[obj])
    break_rows.append(row)
axes.flat[0].legend(fontsize=8)
fig.suptitle("Nuclear OAT across the full lattice — g1 fuel-deletion APPROX caveat",
             fontweight="bold")
fig.tight_layout(); fig.savefig("figs/nuclear_fullrange.png", dpi=150); plt.close(fig)

nuc_break = pd.DataFrame(break_rows)
nuc_break.to_csv("nuclear_break_test.csv", index=False)
print(nuc_break.round(2).to_string(index=False))

                    objective  max_abs_resid_above_0p5  signed_resid_at_1p0  max_resid_over_floor
                load_shed_mwh                  4060.47             -4060.47                  1.35
         true_curtailment_mwh                147264.47            147264.47                 29.45
           total_cost_raw_usd               2676335.07           2676335.07                  5.35
total_cost_less_synthetic_usd               1845368.43           1845368.43                  3.69
        reserve_shortfall_mwh                 15356.95            -15356.95                   NaN
               thermal_starts                   193.94              -169.86                   NaN


## 3. pv top-up points vs the old curve

In [4]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
pv_rows = []
for ax, obj in zip(axes.flat, OBJ_COLS):
    w_o, y_o = pv_old["pv_omega"].to_numpy(), pv_old[obj].to_numpy()
    w_n, y_n = pv_new["pv_omega"].to_numpy(), pv_new[obj].to_numpy()
    b, a = np.polyfit(w_o, y_o, 1)
    grid = np.linspace(0.02, 1.0, 100)
    ax.plot(w_o, y_o, "o-", color="tab:green", label="sweep_C (ω ≤ 0.8)")
    ax.plot(w_n, y_n, "D", color="tab:red", label="top-up (0.88125, 1.0)")
    ax.plot(grid, b * grid + a, ":", color="0.4", label="linear fit ≤ 0.8, extrapolated")
    ax.set_xlabel("pv ω"); ax.set_ylabel(obj, fontsize=8); ax.set_title(obj, fontsize=10)
    resid = y_n - (b * w_n + a)
    row = {"objective": obj, "resid_0p88125": float(resid[0]), "resid_1p0": float(resid[1])}
    if FLOORS[obj]:
        row["max_resid_over_floor"] = float(np.max(np.abs(resid)) / FLOORS[obj])
    pv_rows.append(row)
axes.flat[0].legend(fontsize=8)
fig.suptitle("pv OAT with the extrapolation top-up", fontweight="bold")
fig.tight_layout(); fig.savefig("figs/pv_topup.png", dpi=150); plt.close(fig)
pv_break = pd.DataFrame(pv_rows)
pv_break.to_csv("pv_topup_test.csv", index=False)
print(pv_break.round(2).to_string(index=False))

                    objective  resid_0p88125   resid_1p0  max_resid_over_floor
                load_shed_mwh        3526.45     1622.21                  1.18
         true_curtailment_mwh       59149.12   100792.90                 20.16
           total_cost_raw_usd    -1867725.99 -3373202.84                  6.75
total_cost_less_synthetic_usd    -1656802.66 -2988548.16                  5.98
        reserve_shortfall_mwh        7265.22     3274.48                   NaN
               thermal_starts          10.76       24.76                   NaN


## 4. GP LOO diagnostics on the 16 n₀ points → top-up go/no-go

Pre-registered: during the M-decision dead time, extend n₀ 16 → 32 via
`skip = n_drawn_total` (= 16), ~160 core-h. GP per the bo_replay reference
(`replay.py:182-199`): sklearn Matérn-2.5 ARD, WhiteKernel, `normalize_y=True`,
inputs min-max scaled **per training fold** (per-fold standardization). LOO over
the 16 points; RMSE compared to the noise floor and to the objective's own range.

In [5]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.exceptions import ConvergenceWarning

X = n0[[f"{t}_omega" for t in SORTED_TIERS]].to_numpy(float)   # d = 6, all active
assert not np.isnan(X).any()

def loo_preds(X, y, seed=0):
    n = len(y)
    preds = np.empty(n)
    for i in range(n):
        tr = np.arange(n) != i
        Xtr, ytr = X[tr], y[tr]
        lo = Xtr.min(axis=0); span = np.where((Xtr.max(0) - lo) <= 0, 1.0, Xtr.max(0) - lo)
        kernel = (ConstantKernel(1.0, (1e-3, 1e3))
                  * Matern(length_scale=np.ones(X.shape[1]),
                           length_scale_bounds=(1e-2, 1e1), nu=2.5)
                  + WhiteKernel(1e-4, (1e-6, 1e0)))
        gpr = GaussianProcessRegressor(kernel=kernel, normalize_y=True,
                                       n_restarts_optimizer=1, random_state=seed,
                                       alpha=1e-8)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            gpr.fit((Xtr - lo) / span, ytr)
        preds[i] = gpr.predict(((X[i] - lo) / span).reshape(1, -1))[0]
    return preds

loo_rows = []
for obj in OBJ_COLS:
    y = n0[obj].to_numpy(float)
    p = loo_preds(X, y)
    rmse = float(np.sqrt(np.mean((p - y) ** 2)))
    rng = float(y.max() - y.min())
    row = {"objective": obj, "loo_rmse": rmse, "y_range": rng,
           "rmse_over_range": rmse / rng,
           "loo_R2": float(1 - np.sum((p - y) ** 2) / np.sum((y - y.mean()) ** 2))}
    if FLOORS[obj]:
        row["rmse_over_floor"] = rmse / FLOORS[obj]
    loo_rows.append(row)
loo = pd.DataFrame(loo_rows)
loo.to_csv("n0_gp_loo.csv", index=False)
with pd.option_context("display.width", 160):
    print(loo.round(3).to_string(index=False))
print("\nInterpretation: rmse_over_floor >> 1 ⇒ the GP is model-limited, not "
      "noise-limited — more design points buy real information. loo_R2 near or "
      "below 0 ⇒ 16 points cannot yet support a 6-D surrogate for that objective.")

                    objective    loo_rmse      y_range  rmse_over_range  loo_R2  rmse_over_floor
                load_shed_mwh    1497.157 1.370689e+04            0.109   0.876            0.499
         true_curtailment_mwh   85361.692 5.400735e+05            0.158   0.675           17.072
           total_cost_raw_usd 3619112.550 1.009143e+08            0.036   0.984            7.238
total_cost_less_synthetic_usd 3557638.744 1.019616e+08            0.035   0.985            7.115
        reserve_shortfall_mwh    7011.109 5.131653e+04            0.137   0.831              NaN
               thermal_starts     179.346 1.111000e+03            0.161   0.683              NaN

Interpretation: rmse_over_floor >> 1 ⇒ the GP is model-limited, not noise-limited — more design points buy real information. loo_R2 near or below 0 ⇒ 16 points cannot yet support a 6-D surrogate for that objective.


In [6]:
summary = {
    "collector_commit": "fc98902",
    "n0_rows": 16, "backfill_rows": 10,
    "ranges": ranges.to_dict("records"),
    "nuclear_break": nuc_break.to_dict("records"),
    "pv_topup": pv_break.to_dict("records"),
    "gp_loo": loo.to_dict("records"),
}
with open("t4_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("t4_summary.json written; figures:", sorted(os.listdir("figs")))

t4_summary.json written; figures: ['n0_vs_hull.png', 'nuclear_fullrange.png', 'pv_topup.png']
